# Pre-processing WBM Data
This code cleans up (e.g., renames dimensions and values that were incorrect) WBM datasets such that users can work with clean files that are easier to understand. Furthermore, it compresses the data to reduce data volumes.

In [ ]:
import os

import geopandas as gpd
import numpy as np
import xarray as xr

## Domestic per glacier shapefile data

In [ ]:
process_data = False  # processed 6/23/26

if process_data:
    
    shp_fp = 'wbm_data/domestic_pg_frac/'
    shp_fp_new = 'wbm_data/domestic_pg_frac-compressed/'

    shp_fns = []
    for fn in os.listdir(shp_fp):
        if fn.endswith('_dom_pg.shp'):
            shp_fns.append(fn)
    shp_fns = sorted(shp_fns)

    for fn in shp_fns:
        
        if not os.path.exists(shp_fp_new + fn):

            if fn.startswith('historical'):

                gdf = gpd.read_file(shp_fp + fn)

                gdf = gdf.rename(columns={
                    'domUseSfWt':'1980', 'domUseSfW0':'1981', 'domUseSfW1':'1982', 'domUseSfW2':'1983', 'domUseSfW3':'1984',
                    'domUseSfW4':'1985', 'domUseSfW5':'1986', 'domUseSfW6':'1987', 'domUseSfW7':'1988', 'domUseSfW8':'1989', 
                    'domUseSfW9':'1990', 'domUseSf_1':'1991', 'domUseSf_2':'1992', 'domUseSf_3':'1993', 'domUseSf_4':'1994', 
                    'domUseSf_5':'1995', 'domUseSf_6':'1996', 'domUseSf_7':'1997', 'domUseSf_8':'1998', 'domUseSf_9':'1999', 
                    'domUseSf10':'2000', 'domUseSf11':'2001', 'domUseSf12':'2002', 'domUseSf13':'2003', 'domUseSf14':'2004', 
                    'domUseSf15':'2005', 'domUseSf16':'2006', 'domUseSf17':'2007', 'domUseSf18':'2008', 'domUseSf19':'2009', 
                    'domUseSf20':'2010', 'domUseSf21':'2011', 'domUseSf22':'2012', 'domUseSf23':'2013', 'domUseSf24':'2014', 
                    'domUseSf25':'2015'
                })

            else:

                gdf = gpd.read_file(shp_fp + fn)
                cn_conversion_dict = {f"d_{year}": str(year) for year in range(2016, 2100)}
                gdf = gdf.rename(columns=cn_conversion_dict)

            # Export file
            if not os.path.exists(shp_fp_new):
                os.makedirs(shp_fp_new)
            
            gdf.to_file(shp_fp_new + fn, driver="ESRI Shapefile", index=False)
            

## Discharge monthly max data

In [ ]:
process_data = False   # processed 6/23/2026

if process_data:
    ds_fp = 'wbm_data/discharge_month_max/'
    ds_fp_new = 'wbm_data/discharge_month_max-compressed/'
    
    vn = 'max_monthly_frac'
    
    ds_fns = []
    for fn in os.listdir(ds_fp):
        if fn.endswith('_max_discharge_m3s_pg_frac.nc'):
            ds_fns.append(fn)
    ds_fns = sorted(ds_fns)
    
    for fn in ds_fns:
        
        if not os.path.exists(ds_fp_new + fn):
    
            # Open dataset
            ds = xr.open_dataset(ds_fp + fn)
        
            if not fn.startswith('historical'): # historical file is correctly named, so only needs to be compressed
                # Rename dimension/coordinate
                ds = ds.rename({"Z1": "year"})
            
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2016, 2100))
        
            # Export file
            if not os.path.exists(ds_fp_new):
                os.makedirs(ds_fp_new)
    
            ds.to_netcdf(ds_fp_new + fn, 
                         format='NETCDF4', 
                         encoding={vn: {'zlib': True, 'complevel': 5}})


## Glacier melt data

In [ ]:
process_data = False   # processed 6/23/2026

if process_data:
    ds_fp = 'wbm_data/glMelt/'
    ds_fp_new = 'wbm_data/glMelt-compressed/'
    
    vn = 'glMelt'
    
    ds_fns = []
    for fn in os.listdir(ds_fp):
        if fn.endswith('-glMelt.nc'):
            ds_fns.append(fn)
    ds_fns = sorted(ds_fns)
    
    for fn in ds_fns:
        
        if not os.path.exists(ds_fp_new + fn):
    
            # Open dataset
            ds = xr.open_dataset(ds_fp + fn)
        
            if fn.startswith('historical'):
                ds = ds.drop_vars('glMelt_sigma')
            else:
                # Rename dimension/coordinate
                ds = ds.rename({"time": "year"})
            
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2016, 2100))

            # Export file
            if not os.path.exists(ds_fp_new):
                os.makedirs(ds_fp_new)
    
            ds.to_netcdf(ds_fp_new + fn, 
                         format='NETCDF4', 
                         encoding={vn: {'zlib': True, 'complevel': 5}})

## Irrigation month max data

In [ ]:
process_data = False   # processed 6/23/2026

if process_data:
    ds_fp = 'wbm_data/irr_month_max/'
    ds_fp_new = 'wbm_data/irr_month_max-compressed/'
    
    vn = 'max_monthly_frac'
    
    ds_fns = []
    for fn in os.listdir(ds_fp):
        if fn.endswith('_max_GrossIrr_mm_pg_frac.nc'):
            ds_fns.append(fn)
    ds_fns = sorted(ds_fns)
    
    for fn in ds_fns:
        
        if not os.path.exists(ds_fp_new + fn):
    
            # Open dataset
            ds = xr.open_dataset(ds_fp + fn)
        
            if fn.startswith('historical'): 
                # Rename dimension/coordinate
                ds = ds.rename({"Z1": "year"})
            
                # Assign year values
                ds = ds.assign_coords(year=np.arange(1980, 2016))

            else:
                # Rename dimension/coordinate
                ds = ds.rename({"Z1": "year"})
            
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2016, 2100))

            # Export file
            if not os.path.exists(ds_fp_new):
                os.makedirs(ds_fp_new)
    
            ds.to_netcdf(ds_fp_new + fn, 
                         format='NETCDF4', 
                         encoding={vn: {'zlib': True, 'complevel': 5}})

## Crop Production data

In [ ]:
process_data = False   # processed 6/23/2026

if process_data:
    ds_fp = 'wbm_data/irrProduction_pg/'
    ds_fp_new = 'wbm_data/irrProduction_pg-compressed/'
    
    vn = 'irrProduction_pg'
    
    ds_fns = []
    for fn in os.listdir(ds_fp):
        if fn.endswith('_irrProduction_pg.nc'):
            ds_fns.append(fn)
    ds_fns = sorted(ds_fns)
    
    for fn in ds_fns:
        
        if not os.path.exists(ds_fp_new + fn):
    
            # Open dataset
            ds = xr.open_dataset(ds_fp + fn)

            if fn.startswith('historical'):
                # Assign year values
                ds = ds.assign_coords(year=np.arange(1980, 2016))

            else:
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2016, 2100))

            # Export file
            if not os.path.exists(ds_fp_new):
                os.makedirs(ds_fp_new)
    
            ds.to_netcdf(ds_fp_new + fn, 
                         format='NETCDF4', 
                         encoding={vn: {'zlib': True, 'complevel': 5}})

## Population data

In [ ]:
process_data = False   # processed 6/23/2026

if process_data:
    ds_fp = 'wbm_data/population_gridded/'
    ds_fp_new = 'wbm_data/population_gridded-compressed/'
    
    vn = 'population'
    
    ds_fns = []
    for fn in os.listdir(ds_fp):
        if fn.endswith('_population_per_grid.nc'):
            ds_fns.append(fn)
    ds_fns = sorted(ds_fns)
    
    for fn in ds_fns:
        
        if not os.path.exists(ds_fp_new + fn):
    
            # Open dataset
            ds = xr.open_dataset(ds_fp + fn)

            if fn.startswith('historical'):
                # Rename dimension/coordinate
                ds = ds.rename({"time": "year"})
                
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2000, 2021))

            else:
                # Rename dimension/coordinate
                ds = ds.rename({"time": "year"})
                
                # Assign year values
                ds = ds.assign_coords(year=np.arange(2000, 2101))

            # Export file
            if not os.path.exists(ds_fp_new):
                os.makedirs(ds_fp_new)
    
            ds.to_netcdf(ds_fp_new + fn, 
                         format='NETCDF4', 
                         encoding={vn: {'zlib': True, 'complevel': 5}})